# 04 SHAP Interpretation – Configurable Pipeline

## Purpose

This notebook generates SHAP explanations using the same standardised approach across German Credit, Pima Diabetes and COMPAS.

Only the `ACTIVE_DATASET` value in the configuration block should be changed.

This version includes two important updates:

1. **Reproducibility**: Kernel SHAP background sampling uses `np.random.RandomState(RANDOM_STATE)`.
2. **Reconstruction fidelity**: Tree models are attempted in probability-output mode so Decision Tree, Random Forest and XGBoost can also produce SHAP reconstruction errors.

## Key Design Principle

The notebook explains all test instances and saves outputs in a standard long format for downstream SHAP-LIME disagreement, fidelity, stability and EES scoring.

SHAP reconstruction compares:

```text
model predicted probability
vs
SHAP expected value + sum(SHAP values)
```

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "shap_folder": "04_shap_interpretation_german_credit",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age", "Sex"]
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "shap_folder": "04_shap_interpretation_pima_diabetes",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age"]
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "shap_folder": "04_shap_interpretation_compas",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["age", "sex", "race"]
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]
print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])
print("Positive class:", CONFIG["positive_class_label"])

Active dataset: COMPAS
Dataset name: compas
Positive class: Recidivism


In [2]:
# =========================================================
# 1. Install Required Packages
# =========================================================

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    print("Installing SHAP...")
    !pip install -q shap
    import shap
    SHAP_AVAILABLE = True

print("SHAP available:", SHAP_AVAILABLE)
print("SHAP version:", shap.__version__)

SHAP available: True
SHAP version: 0.52.0


In [3]:
# =========================================================
# 2. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import pandas as pd
import joblib

from scipy.special import expit

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [4]:
# =========================================================
# 3. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["shap_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_MODELS_PATH = BASELINE_RESULTS_PATH / "models"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
FIGURES_PATH = RESULTS_PATH / "figures"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, FIGURES_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Baseline models path:", BASELINE_MODELS_PATH)
print("SHAP output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/compas
Baseline models path: /content/drive/MyDrive/Master_Research/07_Results/compas/03_baseline_modeling_compas/models
SHAP output path: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas


In [5]:
# =========================================================
# 4. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def load_target_series(path):
    target_df = read_csv_safely(path, "target file")
    if "target" in target_df.columns:
        return target_df["target"].astype(int)
    if CONFIG["target_name"] in target_df.columns:
        return target_df[CONFIG["target_name"]].astype(int)
    if target_df.shape[1] == 1:
        return target_df.iloc[:, 0].astype(int)
    raise ValueError(f"Could not identify target column in {path}. Available columns: {target_df.columns.tolist()}")


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def get_model_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return expit(scores)
    preds = model.predict(X)
    return preds.astype(float)


def standardise_shap_values(shap_values, positive_class_index=1):
    values = shap_values
    if hasattr(values, "values"):
        values = values.values
    if isinstance(values, list):
        values = values[positive_class_index] if len(values) > positive_class_index else values[-1]
    values = np.array(values)
    if values.ndim == 3:
        values = values[:, :, positive_class_index] if values.shape[-1] > positive_class_index else values[:, :, -1]
    return values


def get_expected_value(explainer, positive_class_index=1):
    expected_value = explainer.expected_value
    if isinstance(expected_value, list) or isinstance(expected_value, np.ndarray):
        expected_array = np.array(expected_value).flatten()
        return float(expected_array[positive_class_index]) if len(expected_array) > positive_class_index else float(expected_array[-1])
    return float(expected_value)


def classify_model_family(model_id):
    if model_id == "logistic_regression":
        return "linear"
    if model_id in ["decision_tree", "random_forest", "xgboost"]:
        return "tree"
    if model_id == "mlp":
        return "kernel"
    return "unknown"


def make_predict_proba_fn(model, feature_names):
    def predict_fn(data):
        data_df = pd.DataFrame(data, columns=feature_names)
        return model.predict_proba(data_df)[:, 1]
    return predict_fn


def explain_with_kernel_probability(model, X_background, X_explain, feature_names, nsamples):
    predict_fn = make_predict_proba_fn(model, feature_names)
    explainer = shap.KernelExplainer(predict_fn, X_background)
    raw_shap_values = explainer.shap_values(X_explain, nsamples=nsamples)
    shap_values = standardise_shap_values(raw_shap_values)
    expected_value = get_expected_value(explainer)
    return explainer, shap_values, expected_value

## 5. Load Modelling Data

The notebook explains estimator-only models using preprocessed train and test matrices created in `02`.

Tracking columns are removed so SHAP explains only model features.

In [6]:
# =========================================================
# 5. Load Modelling Data
# =========================================================

X_train_preprocessed = read_csv_safely(PREPROCESSING_DATASETS_PATH / "X_train_preprocessed.csv", "X_train_preprocessed")
X_test_preprocessed = read_csv_safely(PREPROCESSING_DATASETS_PATH / "X_test_preprocessed.csv", "X_test_preprocessed")
y_train = load_target_series(PREPROCESSING_DATASETS_PATH / "y_train.csv")
y_test = load_target_series(PREPROCESSING_DATASETS_PATH / "y_test.csv")

PREPROCESSED_TRACKING_COLUMNS = ["original_index", "index", "Unnamed: 0"]

X_train_preprocessed = X_train_preprocessed.drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore").copy()
X_test_preprocessed = X_test_preprocessed.drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore").copy()

feature_names = X_train_preprocessed.columns.tolist()

print("X_train_preprocessed:", X_train_preprocessed.shape)
print("X_test_preprocessed:", X_test_preprocessed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Number of features:", len(feature_names))
display(X_train_preprocessed.head())

X_train_preprocessed: (13996, 30)
X_test_preprocessed: (3500, 30)
y_train: (13996,)
y_test: (3500,)
Number of features: 30


,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_days_from_compas,sex_Female,sex_Male,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other,c_charge_degree_(CO3),c_charge_degree_(CT),c_charge_degree_(F1),c_charge_degree_(F2),c_charge_degree_(F3),c_charge_degree_(F5),c_charge_degree_(F6),c_charge_degree_(F7),c_charge_degree_(M1),c_charge_degree_(M2),c_charge_degree_(MO3),c_charge_degree_(NI0),c_charge_degree_(TCX),c_charge_degree_(X),c_charge_degree_Missing
0,0.375000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.187500,0.000000,0.000000,1.000000,2.600000,-3.000000,4.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,-0.062500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.875000,0.000000,0.000000,0.000000,-0.400000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.125000,0.000000,0.000000,0.000000,2.200000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
# =========================================================
# 6. Validate SHAP Inputs
# =========================================================

validation_rows = [
    {"check": "X_train_preprocessed aligns with y_train", "passed": len(X_train_preprocessed) == len(y_train)},
    {"check": "X_test_preprocessed aligns with y_test", "passed": len(X_test_preprocessed) == len(y_test)},
    {"check": "Train/test columns match", "passed": list(X_train_preprocessed.columns) == list(X_test_preprocessed.columns)},
    {"check": "No missing train values", "passed": X_train_preprocessed.isna().sum().sum() == 0},
    {"check": "No missing test values", "passed": X_test_preprocessed.isna().sum().sum() == 0},
    {"check": "Binary test target", "passed": sorted(pd.Series(y_test).unique().tolist()) == [0, 1]},
]

shap_input_validation_df = pd.DataFrame(validation_rows)
display(shap_input_validation_df)
save_df(shap_input_validation_df, "shap_input_validation.csv")

if not shap_input_validation_df["passed"].all():
    raise ValueError("One or more SHAP input validation checks failed.")

,check,passed
0,X_train_preprocessed aligns with y_train,True
1,X_test_preprocessed aligns with y_test,True
2,Train/test columns match,True
3,No missing train values,True
4,No missing test values,True
5,Binary test target,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_input_validation.csv


## 7. Load Estimator-Only Models

SHAP explains the estimator-only models saved by `03_baseline_modeling`.

In [8]:
# =========================================================
# 7. Load Estimator-Only Models
# =========================================================

model_files = {
    "logistic_regression": "logistic_regression.pkl",
    "decision_tree": "decision_tree.pkl",
    "random_forest": "random_forest.pkl",
    "xgboost": "xgboost.pkl",
    "mlp": "mlp.pkl",
}

pretty_model_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "mlp": "MLP",
}

models = {}
model_load_rows = []

for model_id, filename in model_files.items():
    model_path = BASELINE_MODELS_PATH / filename
    if model_path.exists():
        models[model_id] = joblib.load(model_path)
        found = True
        status = "Loaded"
    else:
        found = False
        status = "Skipped - model file not found"

    model_load_rows.append({
        "model_id": model_id,
        "model_name": pretty_model_names[model_id],
        "model_file_found": found,
        "model_file_path": str(model_path),
        "status": status
    })

model_load_registry_df = pd.DataFrame(model_load_rows)
display(model_load_registry_df)
save_df(model_load_registry_df, "shap_model_load_registry.csv")

if not models:
    raise ValueError("No estimator-only models were loaded.")

,model_id,model_name,model_file_found,model_file_path,status
0,logistic_regression,Logistic Regression,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
1,decision_tree,Decision Tree,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
2,random_forest,Random Forest,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
3,xgboost,XGBoost,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
4,mlp,MLP,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_model_load_registry.csv


## 8. SHAP Configuration

`KERNEL_NSAMPLES` controls Kernel SHAP approximation quality. It does **not** limit the number of test instances explained.

All available test instances are explained.

In [9]:
# =========================================================
# 8. SHAP Configuration
# =========================================================

KERNEL_BACKGROUND_SIZE = min(50, len(X_train_preprocessed))
KERNEL_NSAMPLES = 500
USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY = True

shap_config_df = pd.DataFrame([
    {"parameter": "RANDOM_STATE", "value": RANDOM_STATE, "description": "Fixed seed for reproducible Kernel SHAP background sampling."},
    {"parameter": "KERNEL_BACKGROUND_SIZE", "value": KERNEL_BACKGROUND_SIZE, "description": "Background sample size used by Kernel SHAP."},
    {"parameter": "KERNEL_NSAMPLES", "value": KERNEL_NSAMPLES, "description": "Coalition samples per explained instance for Kernel SHAP."},
    {"parameter": "USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY", "value": USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY, "description": "Fallback to Kernel SHAP if probability-scale Tree SHAP is unavailable."},
    {"parameter": "explanation_population", "value": "all_test_instances", "description": "SHAP explanations generated for all test instances."}
])

display(shap_config_df)
save_df(shap_config_df, "shap_config.csv")

,parameter,value,description
0,RANDOM_STATE,42,Fixed seed for reproducible Kernel SHAP backgr...
1,KERNEL_BACKGROUND_SIZE,50,Background sample size used by Kernel SHAP.
2,KERNEL_NSAMPLES,500,Coalition samples per explained instance for K...
3,USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY,True,Fallback to Kernel SHAP if probability-scale T...
4,explanation_population,all_test_instances,SHAP explanations generated for all test insta...


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_config.csv')

## 9. Generate All-Test SHAP Explanations

Tree models are first attempted using probability-scale Tree SHAP. If unsupported, the notebook falls back to Kernel SHAP probability explanations so reconstruction errors can still be calculated.

In [10]:
# =========================================================
# 9. Generate All-Test SHAP Explanations
# =========================================================

X_train_values = X_train_preprocessed.values
X_test_values = X_test_preprocessed.values

background_indices = rng.choice(
    len(X_train_preprocessed),
    size=KERNEL_BACKGROUND_SIZE,
    replace=False
)

kernel_background = X_train_values[background_indices]

all_shap_rows = []
prediction_rows = []
global_importance_rows = []
reconstruction_rows = []
error_rows = []
explainer_diagnostic_rows = []

for model_id, model in models.items():
    model_name = pretty_model_names[model_id]
    model_family = classify_model_family(model_id)

    print("=" * 100)
    print(f"Generating SHAP explanations for: {model_name}")
    print("=" * 100)

    start_time = time.time()

    try:
        y_pred = model.predict(X_test_preprocessed)
        y_proba = get_model_probabilities(model, X_test_preprocessed)

        explainer_type = None
        fallback_used = False
        fallback_reason = None

        if model_family == "linear":
            explainer = shap.LinearExplainer(model, X_train_preprocessed)
            raw_shap_values = explainer.shap_values(X_test_preprocessed)
            shap_values = standardise_shap_values(raw_shap_values)
            expected_value = get_expected_value(explainer)
            reconstruction_scale = "log_odds"
            reconstructed_raw = expected_value + shap_values.sum(axis=1)
            reconstructed_probability = expit(reconstructed_raw)
            reconstruction_error = np.abs(y_proba - reconstructed_probability)
            explainer_type = "LinearExplainer"

        elif model_family == "tree":
            try:
                explainer = shap.TreeExplainer(
                    model,
                    data=X_train_preprocessed,
                    model_output="probability",
                    feature_perturbation="interventional"
                )
                raw_shap_values = explainer.shap_values(X_test_preprocessed)
                shap_values = standardise_shap_values(raw_shap_values)
                expected_value = get_expected_value(explainer)
                reconstruction_scale = "probability"
                reconstructed_raw = expected_value + shap_values.sum(axis=1)
                reconstructed_probability = reconstructed_raw
                reconstruction_error = np.abs(y_proba - reconstructed_probability)
                explainer_type = "TreeExplainer_probability_interventional"

            except Exception as tree_probability_error:
                fallback_reason = str(tree_probability_error)
                if not USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY:
                    raise tree_probability_error

                print(f"Probability-scale Tree SHAP failed for {model_name}. Falling back to Kernel SHAP probability explanations.")
                explainer, shap_values, expected_value = explain_with_kernel_probability(
                    model=model,
                    X_background=kernel_background,
                    X_explain=X_test_values,
                    feature_names=feature_names,
                    nsamples=KERNEL_NSAMPLES
                )
                reconstruction_scale = "probability"
                reconstructed_raw = expected_value + shap_values.sum(axis=1)
                reconstructed_probability = reconstructed_raw
                reconstruction_error = np.abs(y_proba - reconstructed_probability)
                explainer_type = "KernelExplainer_probability_tree_fallback"
                fallback_used = True

        elif model_family == "kernel":
            explainer, shap_values, expected_value = explain_with_kernel_probability(
                model=model,
                X_background=kernel_background,
                X_explain=X_test_values,
                feature_names=feature_names,
                nsamples=KERNEL_NSAMPLES
            )
            reconstruction_scale = "probability"
            reconstructed_raw = expected_value + shap_values.sum(axis=1)
            reconstructed_probability = reconstructed_raw
            reconstruction_error = np.abs(y_proba - reconstructed_probability)
            explainer_type = "KernelExplainer_probability"

        else:
            raise ValueError(f"Unsupported SHAP model family: {model_family}")

        elapsed_seconds = time.time() - start_time

        explainer_diagnostic_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "model_family": model_family,
            "explainer_type": explainer_type,
            "reconstruction_scale": reconstruction_scale,
            "fallback_used": fallback_used,
            "fallback_reason": fallback_reason,
            "elapsed_seconds": elapsed_seconds,
            "n_test_instances_explained": len(X_test_preprocessed),
            "n_features": len(feature_names)
        })

        for i in range(len(X_test_preprocessed)):
            prediction_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": i,
                "true_class": int(y_test.iloc[i] if hasattr(y_test, "iloc") else y_test[i]),
                "predicted_class": int(y_pred[i]),
                "predicted_probability": float(y_proba[i])
            })

            reconstruction_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": i,
                "expected_value": float(expected_value),
                "shap_sum": float(shap_values[i].sum()),
                "reconstructed_raw_output": float(reconstructed_raw[i]),
                "reconstructed_probability": float(reconstructed_probability[i]),
                "predicted_probability": float(y_proba[i]),
                "reconstruction_error": float(reconstruction_error[i]),
                "reconstruction_scale": reconstruction_scale,
                "explainer_type": explainer_type,
                "fallback_used": fallback_used
            })

        for i in range(len(X_test_preprocessed)):
            for j, feature in enumerate(feature_names):
                all_shap_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": i,
                    "feature": feature,
                    "shap_value": float(shap_values[i, j]),
                    "abs_shap_value": float(abs(shap_values[i, j])),
                    "feature_value": float(X_test_preprocessed.iloc[i, j]),
                    "explainer_type": explainer_type
                })

        mean_abs_values = np.abs(shap_values).mean(axis=0)
        for feature, value in zip(feature_names, mean_abs_values):
            global_importance_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "feature": feature,
                "mean_abs_shap_value": float(value),
                "explainer_type": explainer_type
            })

        print(f"Completed {model_name} in {elapsed_seconds:.2f} seconds using {explainer_type}.")

    except Exception as e:
        error_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "error_message": str(e)
        })
        print(f"SHAP failed for {model_name}: {e}")

all_test_local_shap_long_df = pd.DataFrame(all_shap_rows)
all_test_prediction_summary_df = pd.DataFrame(prediction_rows)
shap_reconstruction_diagnostics_df = pd.DataFrame(reconstruction_rows)
shap_error_log_df = pd.DataFrame(error_rows)
global_shap_importance_summary_df = pd.DataFrame(global_importance_rows)
shap_explainer_diagnostics_df = pd.DataFrame(explainer_diagnostic_rows)

if not global_shap_importance_summary_df.empty:
    global_shap_importance_summary_df = global_shap_importance_summary_df.sort_values(
        ["model_id", "mean_abs_shap_value"], ascending=[True, False]
    ).reset_index(drop=True)

print("Local SHAP shape:", all_test_local_shap_long_df.shape)
print("Prediction summary shape:", all_test_prediction_summary_df.shape)
print("Reconstruction diagnostics shape:", shap_reconstruction_diagnostics_df.shape)
print("Global importance shape:", global_shap_importance_summary_df.shape)
print("Explainer diagnostics shape:", shap_explainer_diagnostics_df.shape)
print("Error log shape:", shap_error_log_df.shape)

display(shap_explainer_diagnostics_df)
display(all_test_local_shap_long_df.head())
display(global_shap_importance_summary_df.head())
display(shap_error_log_df)

Generating SHAP explanations for: Logistic Regression


Completed Logistic Regression in 0.01 seconds using LinearExplainer.
Generating SHAP explanations for: Decision Tree
Completed Decision Tree in 0.41 seconds using TreeExplainer_probability_interventional.
Generating SHAP explanations for: Random Forest


100%|===================| 6998/7000 [32:42<00:00]       WARNING:shap:Background dataset has 13996 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=13996 when initializing the masker.


Completed Random Forest in 1962.88 seconds using TreeExplainer_probability_interventional.
Generating SHAP explanations for: XGBoost


 96%|=================== | 3360/3500 [00:14<00:00]       

...

Completed MLP in 175.29 seconds using KernelExplainer_probability.
Local SHAP shape: (525000, 8)
Prediction summary shape: (17500, 6)
Reconstruction diagnostics shape: (17500, 12)
Global importance shape: (150, 5)
Explainer diagnostics shape: (5, 10)
Error log shape: (0, 0)


,model_id,model_name,model_family,explainer_type,reconstruction_scale,fallback_used,fallback_reason,elapsed_seconds,n_test_instances_explained,n_features
0,logistic_regression,Logistic Regression,linear,LinearExplainer,log_odds,False,None,0.010015,3500,30
1,decision_tree,Decision Tree,tree,TreeExplainer_probability_interventional,probability,False,None,0.414641,3500,30
2,random_forest,Random Forest,tree,TreeExplainer_probability_interventional,probability,False,None,1962.877854,3500,30
3,xgboost,XGBoost,tree,TreeExplainer_probability_interventional,probability,False,None,14.380127,3500,30
4,mlp,MLP,kernel,KernelExplainer_probability,probability,False,None,175.292113,3500,30


,model_id,model_name,test_instance_position,feature,shap_value,abs_shap_value,feature_value,explainer_type
0,logistic_regression,Logistic Regression,0,age,0.370608,0.370608,-0.437500,LinearExplainer
1,logistic_regression,Logistic Regression,0,juv_fel_count,-0.002708,0.002708,0.000000,LinearExplainer
2,logistic_regression,Logistic Regression,0,juv_misd_count,-0.007029,0.007029,0.000000,LinearExplainer
3,logistic_regression,Logistic Regression,0,juv_other_count,0.113646,0.113646,1.000000,LinearExplainer
4,logistic_regression,Logistic Regression,0,priors_count,-0.173139,0.173139,0.000000,LinearExplainer


,model_id,model_name,feature,mean_abs_shap_value,explainer_type
0,decision_tree,Decision Tree,priors_count,0.134505,TreeExplainer_probability_interventional
1,decision_tree,Decision Tree,age,0.085312,TreeExplainer_probability_interventional
2,decision_tree,Decision Tree,days_b_screening_arrest,0.018927,TreeExplainer_probability_interventional
3,decision_tree,Decision Tree,c_days_from_compas,0.015473,TreeExplainer_probability_interventional
4,decision_tree,Decision Tree,sex_Female,0.009460,TreeExplainer_probability_interventional


""


## 10. SHAP Reconstruction Diagnostic Summary

A reconstruction error close to zero does not mean the model is perfect. It means the SHAP decomposition reconstructs the model's own predicted probability.

In [11]:
# =========================================================
# 10. SHAP Reconstruction Diagnostic Summary
# =========================================================

if shap_reconstruction_diagnostics_df.empty:
    shap_reconstruction_diagnostic_summary_df = pd.DataFrame()
else:
    shap_reconstruction_diagnostic_summary_df = (
        shap_reconstruction_diagnostics_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_rows=("test_instance_position", "count"),
            n_reconstruction_error_available=("reconstruction_error", lambda x: int(x.notna().sum())),
            n_reconstructed_probability_available=("reconstructed_probability", lambda x: int(x.notna().sum())),
            n_reconstructed_raw_available=("reconstructed_raw_output", lambda x: int(x.notna().sum())),
            mean_reconstruction_error=("reconstruction_error", "mean"),
            median_reconstruction_error=("reconstruction_error", "median"),
            max_reconstruction_error=("reconstruction_error", "max"),
            reconstruction_scales=("reconstruction_scale", lambda x: ", ".join(sorted(set(x.astype(str))))),
            explainer_types=("explainer_type", lambda x: ", ".join(sorted(set(x.astype(str))))),
            fallback_used=("fallback_used", "max")
        )
    )

    shap_reconstruction_diagnostic_summary_df["reconstruction_error_available_pct"] = (
        shap_reconstruction_diagnostic_summary_df["n_reconstruction_error_available"] /
        shap_reconstruction_diagnostic_summary_df["n_rows"] * 100
    )

display(shap_reconstruction_diagnostic_summary_df)
save_df(shap_reconstruction_diagnostic_summary_df, "shap_reconstruction_diagnostic_summary.csv")
save_df(shap_explainer_diagnostics_df, "shap_explainer_diagnostics.csv")

,model_id,model_name,n_rows,n_reconstruction_error_available,n_reconstructed_probability_available,n_reconstructed_raw_available,mean_reconstruction_error,median_reconstruction_error,max_reconstruction_error,reconstruction_scales,explainer_types,fallback_used,reconstruction_error_available_pct
0,decision_tree,Decision Tree,3500,3500,3500,3500,0.000000,0.000000,0.000000,probability,TreeExplainer_probability_interventional,False,100.000000
1,logistic_regression,Logistic Regression,3500,3500,3500,3500,0.000000,0.000000,0.000000,log_odds,LinearExplainer,False,100.000000
2,mlp,MLP,3500,3500,3500,3500,0.000000,0.000000,0.000000,probability,KernelExplainer_probability,False,100.000000
3,random_forest,Random Forest,3500,3500,3500,3500,0.000556,0.000067,0.149933,probability,TreeExplainer_probability_interventional,False,100.000000
4,xgboost,XGBoost,3500,3500,3500,3500,0.000000,0.000000,0.000000,probability,TreeExplainer_probability_interventional,False,100.000000


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_reconstruction_diagnostic_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_explainer_diagnostics.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_explainer_diagnostics.csv')

## 11. Save SHAP Outputs

In [12]:
# =========================================================
# 11. Save SHAP Outputs
# =========================================================

save_df(all_test_local_shap_long_df, "all_test_local_shap_long.csv")
save_df(all_test_prediction_summary_df, "all_test_prediction_summary.csv")
save_df(shap_reconstruction_diagnostics_df, "shap_reconstruction_diagnostics.csv")
save_df(shap_reconstruction_diagnostic_summary_df, "shap_reconstruction_diagnostic_summary.csv")
save_df(global_shap_importance_summary_df, "global_shap_importance_summary.csv")
save_df(shap_explainer_diagnostics_df, "shap_explainer_diagnostics.csv")
save_df(shap_error_log_df, "shap_error_log.csv")

save_df(all_test_local_shap_long_df, "all_test_local_shap_long.csv", folder=DATASETS_PATH)
save_df(all_test_prediction_summary_df, "all_test_prediction_summary.csv", folder=DATASETS_PATH)
save_df(shap_reconstruction_diagnostics_df, "shap_reconstruction_diagnostics.csv", folder=DATASETS_PATH)
save_df(shap_reconstruction_diagnostic_summary_df, "shap_reconstruction_diagnostic_summary.csv", folder=DATASETS_PATH)
save_df(global_shap_importance_summary_df, "global_shap_importance_summary.csv", folder=DATASETS_PATH)
save_df(shap_explainer_diagnostics_df, "shap_explainer_diagnostics.csv", folder=DATASETS_PATH)
save_df(shap_error_log_df, "shap_error_log.csv", folder=DATASETS_PATH)

save_df(all_test_local_shap_long_df, "shap_all_test_long.csv")
save_df(all_test_local_shap_long_df, "shap_all_test_long.csv", folder=DATASETS_PATH)
save_df(shap_reconstruction_diagnostics_df, "shap_fidelity_inputs.csv")
save_df(shap_reconstruction_diagnostics_df, "shap_fidelity_inputs.csv", folder=DATASETS_PATH)

save_excel(
    {
        "all_test_local_shap_long": all_test_local_shap_long_df,
        "prediction_summary": all_test_prediction_summary_df,
        "reconstruction_diagnostics": shap_reconstruction_diagnostics_df,
        "reconstruction_summary": shap_reconstruction_diagnostic_summary_df,
        "global_importance": global_shap_importance_summary_df,
        "explainer_diagnostics": shap_explainer_diagnostics_df,
        "model_load_registry": model_load_registry_df,
        "input_validation": shap_input_validation_df,
        "shap_config": shap_config_df,
        "error_log": shap_error_log_df
    },
    f"shap_interpretation_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "positive_class_label": CONFIG["positive_class_label"],
    "negative_class_label": CONFIG["negative_class_label"],
    "explanation_population": "all_test_instances",
    "kernel_background_size": KERNEL_BACKGROUND_SIZE,
    "kernel_nsamples": KERNEL_NSAMPLES,
    "tree_reconstruction_approach": "TreeExplainer model_output='probability' with interventional perturbation",
    "tree_fallback": "KernelExplainer probability fallback enabled" if USE_KERNEL_FALLBACK_FOR_TREE_PROBABILITY else "No fallback enabled",
    "models_loaded": list(models.keys())
}

with open(LOGS_PATH / "shap_interpretation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "shap_interpretation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All SHAP outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/all_test_local_shap_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/all_test_prediction_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_reconstruction_diagnostics.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_reconstruction_diagnostic_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/global_shap_importance_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_explainer_diagnostics.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables/shap_error_log.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/04_s

## 12. Interpretation Note

This notebook provides SHAP explanation evidence required for the EES framework.

The reconstruction diagnostic is a **SHAP decomposition fidelity check**, not predictive accuracy.

A reconstruction error close to zero means:

```text
SHAP expected value + SHAP feature contributions ≈ model predicted probability
```

LIME provides a complementary local surrogate fidelity measure.

In [13]:
# =========================================================
# 12. Final Completion Message
# =========================================================

print("=" * 100)
print(f"04 SHAP Interpretation completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("Key outputs:")
print("- all_test_local_shap_long.csv")
print("- shap_all_test_long.csv")
print("- all_test_prediction_summary.csv")
print("- shap_reconstruction_diagnostics.csv")
print("- shap_reconstruction_diagnostic_summary.csv")
print("- shap_explainer_diagnostics.csv")
print("- shap_fidelity_inputs.csv")
print("- global_shap_importance_summary.csv")
print(f"- shap_interpretation_{DATASET_NAME}_outputs.xlsx")

04 SHAP Interpretation completed successfully for: COMPAS
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/logs
Key outputs:
- all_test_local_shap_long.csv
- shap_all_test_long.csv
- all_test_prediction_summary.csv
- shap_reconstruction_diagnostics.csv
- shap_reconstruction_diagnostic_summary.csv
- shap_explainer_diagnostics.csv
- shap_fidelity_inputs.csv
- global_shap_importance_summary.csv
- shap_interpretation_compas_outputs.xlsx
